<a href="https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. The contract — five plain-word answers

**What one row means:** One row in my working feature frame represents **one pseudonymized content item for one client at the end of the March 2026 decision month**. The ranking question is which content items should be reviewed first for a possible refresh.

**Tables used:** I use `fact_content_daily_performance` for daily search-performance signals and `dim_clients` for client history/availability checks. I do **not** use `fact_content_daily_performance_sample` for label development because it is the final June 2026 month and must remain sealed.

**Time window:** March 2026 is the development decision window. The five features summarize information available during March. April 2026 supplies an observed next-month outcome for the development/leakage demonstration. June 2026 remains a sealed test month.

**What I predict/rank:** My lane is **ranking/scoring refresh candidates**. The observed outcome is whether April impressions are more than 20% below March impressions, among content items with measurable GSC data in both months. This is an observed outcome, not a claim that a refresh causes recovery.

**One deliberate exclusion:** `trend_pct` / `trend_direction` from a snapshot are excluded from the honest feature set because they are derived from recent movement and can encode the outcome signal. I deliberately add one future movement column later to demonstrate leakage, then remove it.


In [1]:
# Setup: authenticate through a Colab Secret — never paste the token into this notebook.
%pip -q install duckdb huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

# In Colab: click the 🔑 Secrets icon → add a secret named HF_TOKEN → enable notebook access.
# The token value is stored by Colab and is NOT written anywhere in this notebook.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "This notebook expects Google Colab. Add your Hugging Face READ token as a Colab Secret named HF_TOKEN, enable notebook access, then Run All again."
    )

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. In Colab, open Secrets (🔑), create HF_TOKEN, paste the READ token there, enable notebook access, then rerun."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# The secret is created at runtime from the protected Colab Secret; the token is never stored in a cell.
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
print("Authenticated through Colab Secret HF_TOKEN. Token value is not displayed or stored in the notebook.")


Authenticated through Colab Secret HF_TOKEN. Token value is not displayed or stored in the notebook.


## 2. Field classification

### Features — safe at the March decision moment
1. **`impressions_march`** — total GSC impressions during March; knowable when the March window closes.
2. **`clicks_march`** — total GSC clicks during March; knowable when the March window closes.
3. **`avg_position_march`** — average GSC position observed during March; knowable from March observations.
4. **`days_with_impressions_march`** — number of March days with at least one GSC impression; knowable from March observations.
5. **`ctr_march`** — March clicks / March impressions × 100; calculated only from March data.

### Label / observed outcome
`declined_next_month` — 1 when April impressions are more than 20% below March impressions, restricted to items with measurable GSC data in both months. It is an observed future outcome, not a causal treatment effect.

### Context
- `client_hash_id` and `content_hash_id` — pseudonymous identifiers for grouping/joining only; never model features.
- `report_date` — used to enforce windows; not a model feature.

### Excluded
- `trend_pct` / `trend_direction` — recent movement/label-derived signals; can leak the outcome.
- `gsc_data_available` — availability/control field, used for filtering, not a feature.
- `ga4_*` — omitted from this first GSC-only frame so unavailable analytics are not treated as zero.
- Provider/model/product-decision fields — not part of the first page-performance feature frame.


In [2]:
# Verification query 1 — grain on the mid-panel March partition.
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Verification 1 — duplicate grain rows:")
display(q1)
assert q1.empty, "Grain check failed: duplicate daily rows found."
print("PASS: no duplicate rows at report_date × client × content grain in March 2026.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verification 1 — duplicate grain rows:


,report_date,client_hash_id,content_hash_id,c


PASS: no duplicate rows at report_date × client × content grain in March 2026.


## 3. Three verification queries

These are intentionally small checks on the **mid-panel March 2026 partition**, not the final `_sample` month. They verify the row grain, slice size/date span, and availability before the feature frame is built.


In [3]:
# Verification query 2 — row count and date span for the March slice.
q2 = con.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MAR}
""").df()
display(q2)


,rows,clients,content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


# Verification query 3 — availability, using IS TRUE as required.
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS not_gsc_available_rows
    FROM {FACT_MAR}
""").df()
display(q3)
print("The feature frame keeps only rows where gsc_data_available IS TRUE.")


In [11]:
## 4. Five-feature frame + deliberate leakage experiment

### Honest five-feature frame from March
# The feature frame is built from the March mid-panel partition. It is aggregated to one row per client content item and uses only March information at the decision moment.
# Feature-frame query (not one of the three verification queries): aggregate March to content grain.
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_march,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
             ELSE NULL END AS ctr_march
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 100
""").df()

# Observed April outcome. It is not included in the honest feature vector.
outcomes = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {FACT_APR}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
""").df()

frame = features.merge(outcomes, on=["client_hash_id", "content_hash_id"], how="inner")
frame["declined_next_month"] = (frame["impressions_april"] < 0.80 * frame["impressions_march"]).astype(int)
FEATURES = ["impressions_march", "clicks_march", "avg_position_march", "days_with_impressions_march", "ctr_march"]
print(f"Five-feature frame: {len(frame):,} items with March features and an observed April outcome.")
display(frame[FEATURES + ["declined_next_month"]].head(10))
### Feature availability notes

# - `impressions_march` - knowable at the decision moment because the March reporting window has closed.
# - `clicks_march` - knowable at the decision moment because March clicks are already observed.
# - `avg_position_march` - knowable at the decision moment because it is computed from March GSC observations.
# - `days_with_impressions_march` - knowable at the decision moment because it counts observed March days.
# - `ctr_march` - knowable at the decision moment because it is calculated only from March impressions and clicks.

# The April outcome is deliberately **not** a feature.

### Deliberate leakage experiment

# The next cell intentionally adds a future April-vs-March movement column. This is a label-derived/future feature and is invalid for the real decision moment. The experiment is only to make the leakage lesson visible, then the column is removed.
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

def precision_at_k(scores, y, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y)[order].mean())

y = frame["declined_next_month"].to_numpy()

# Honest diagnostic: five March-only features.
honest_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=3, random_state=42))
honest_model.fit(frame[FEATURES], y)
honest_scores = honest_model.predict_proba(frame[FEATURES])[:, 1]
honest_p50 = precision_at_k(honest_scores, y, 50)

# Deliberate leak: uses April, the outcome period.
frame["LEAK_future_impression_change_pct"] = ((frame["impressions_april"] - frame["impressions_march"]) / frame["impressions_march"]) * 100.0
leaky_features = FEATURES + ["LEAK_future_impression_change_pct"]
leaky_model = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=3, random_state=42))
leaky_model.fit(frame[leaky_features], y)
leaky_scores = leaky_model.predict_proba(frame[leaky_features])[:, 1]
leaky_p50 = precision_at_k(leaky_scores, y, 50)

print(f"Honest five-feature diagnostic Precision@50: {honest_p50:.3f}")
print(f"Leaky diagnostic Precision@50:              {leaky_p50:.3f}")
print("The leaky score is invalid because it uses future outcome-period information.")

# Remove the trap before finalizing the feature vector.
frame = frame.drop(columns=["LEAK_future_impression_change_pct"])
final_feature_frame = frame[["client_hash_id", "content_hash_id"] + FEATURES + ["declined_next_month"]].copy()
print("Leak removed. Final feature columns:", FEATURES)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Five-feature frame: 100,893 items with March features and an observed April outcome.


,impressions_march,clicks_march,avg_position_march,days_with_impressions_march,ctr_march,declined_next_month
0,1140.0,2.0,4.394234,31,0.175439,0
1,149.0,0.0,8.454069,30,0.000000,1
2,1421.0,6.0,6.320337,31,0.422238,0
3,2770.0,16.0,4.459107,31,0.577617,0
4,150.0,1.0,7.046534,30,0.666667,1
5,6048.0,23.0,4.950311,31,0.380291,0
6,223.0,0.0,52.127896,30,0.000000,1
7,357.0,1.0,20.926174,30,0.280112,1
8,132.0,1.0,33.568489,29,0.757576,1
9,281.0,1.0,36.211416,30,0.355872,1


Honest five-feature diagnostic Precision@50: 0.380
Leaky diagnostic Precision@50:              1.000
The leaky score is invalid because it uses future outcome-period information.
Leak removed. Final feature columns: ['impressions_march', 'clicks_march', 'avg_position_march', 'days_with_impressions_march', 'ctr_march']


## 5. Data limits

**Named limitation:** this March development slice does not establish that refreshing a page causes recovery. It only gives an observed next-month performance outcome for ranking/validation work. The warehouse is an unbalanced panel, so later multi-month work must check `dim_clients.gsc_data_start` and avoid treating missing history as zero. The final June 2026 month is sealed and must not be used while developing the label or feature logic.

Other limits: items without measurable GSC data in both the March feature window and April outcome window are not included in this development outcome set. GA4 is intentionally omitted from the first feature frame.

## Self-check

- [x] Five plain-word contract answers are filled.
- [x] Exactly three verification queries are shown for grain, row/date span, and availability.
- [x] Availability is filtered with `IS TRUE`.
- [x] A five-feature frame is built from the March mid-panel slice.
- [x] Every feature has an explicit “knowable at the decision moment because…” line.
- [x] A deliberate future/label-derived leakage experiment is shown, then the leaked column is removed.
- [x] One named limitation is stated.
- [x] The `_sample` / June 2026 final month is not used for development label logic.
- [ ] Run top-to-bottom in Colab using the `HF_TOKEN` Secret.
- [ ] Confirm all outputs are visible and sensible.
- [ ] Commit as `work/notebooks/w03_data_contract.ipynb` and submit the repo URL.
